# Lab 5A: Model comparison harness
**Claude Certified Developer (Foundations) · 30-Hour Program for UST · Day 5 · about 30 minutes**

**Objectives**
- Write a quality bar before testing
- Run the same labelled set on several model tiers
- Measure accuracy, latency percentiles, and tokens
- Compute cost per request from usage and current prices
- Choose the cheapest model that passes

**Before you start:** in Colab, click the key icon on the left, add a secret named `ANTHROPIC_API_KEY`, and turn on notebook access. All data in this lab is synthetic.

In [ ]:
# Setup: run once per session
!pip -q install -U anthropic 
import os, json, time, random, importlib, re
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in your environment"
import anthropic
print("anthropic SDK", anthropic.__version__, "| key loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))

In [ ]:
%%writefile config.py
"""One place for Claude settings (same pattern as Day 1)."""
import os
# Confirm current IDs on the models overview page: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL = os.environ.get("CLAUDE_MODEL", "claude-sonnet-5")
FAST_MODEL = os.environ.get("CLAUDE_FAST_MODEL", "claude-haiku-4-5-20251001")
MAX_TOKENS = 1024
MAX_RETRIES = 2
TIMEOUT_S = 120

In [ ]:
import config
importlib.reload(config)
client = anthropic.Anthropic(max_retries=config.MAX_RETRIES, timeout=config.TIMEOUT_S)

def log_response(resp, label=""):
    kinds = [b.type for b in resp.content]
    print(f"{label} stop_reason={resp.stop_reason} blocks={kinds} "
          f"input_tokens={resp.usage.input_tokens} output_tokens={resp.usage.output_tokens}")
    for b in resp.content:
        if b.type == "text":
            print("   ", b.text[:300].replace("\n", " "))

def text_of(resp):
    return "".join(b.text for b in resp.content if b.type == "text")

print("model:", config.MODEL, "| fast model:", config.FAST_MODEL)

## 1. Write the quality bar first
Decide what 'good enough' means before you see any results.

In [ ]:
QUALITY_BAR = {"category_accuracy": 0.85, "p95_latency_s": 4.0}
MODELS = [config.FAST_MODEL, config.MODEL]          # add a stronger tier if you like, e.g. "claude-opus-5-5"
print("bar:", QUALITY_BAR, "| models:", MODELS)

In [ ]:
# Synthetic labelled tickets (training data only). Hard cases are marked with a star in the comments.
EVAL = [
 ("Whole sector 9 has had no fiber since 5am, a clinic is affected", "outage", "P1"),
 ("I was charged twice for September", "billing", "P2"),
 ("WiFi light blinks orange after a power cut", "device", "P3"),
 ("How do I change the email on my account?", "account", "P3"),
 ("Mobile data stopped working for everyone in our office block", "outage", "P1"),   # star: 'office' sounds like one customer
 ("Refund the router I returned last week, it's been 10 days", "billing", "P2"),
 ("Router keeps rebooting every hour since the firmware update", "device", "P2"),
 ("I'm locked out after too many password attempts", "account", "P2"),
 ("Is there a student discount?", "other", "P3"),
 ("Calls drop when I walk past the station, only on my phone", "device", "P3"),    # star: sounds like an outage
 ("Your invoice shows a roaming fee but I never left the city", "billing", "P2"),
 ("Can I transfer my plan to my brother?", "account", "P3"),
 ("Whole street lost TV and internet during the storm", "outage", "P1"),
 ("The app crashes when I open bill history", "other", "P3"),                      # star: billing words, app issue
 ("Please close my account at the end of the month", "account", "P3"),
 ("Speed test shows 5 Mbps on a 300 Mbps plan for a week", "device", "P2"),         # star: could be outage
]
print(len(EVAL), "labelled tickets")

In [ ]:
SCHEMA = {"type": "object", "properties": {
    "category": {"type": "string", "enum": ["billing", "outage", "device", "account", "other"]},
    "priority": {"type": "string", "enum": ["P1", "P2", "P3"]},
    "confidence": {"type": "number", "description": "0 to 1"}},
  "required": ["category", "priority", "confidence"], "additionalProperties": False}
SYSTEM = ("Classify telecom support tickets. outage = service down for many customers or an area. device = one customer's equipment or line. "
          "P1 = outage affecting many users or critical services; P2 = one customer blocked or charged wrongly; P3 = everything else.")
def classify(model, text, system=SYSTEM, max_tokens=200):
    t0 = time.perf_counter()
    r = client.messages.create(model=model, max_tokens=max_tokens, system=system,
                               output_config={"format": {"type": "json_schema", "schema": SCHEMA}},
                               messages=[{"role": "user", "content": text}])
    return json.loads(text_of(r)), time.perf_counter() - t0, r.usage

## 2. Run every model on the same examples

In [ ]:
import numpy as np
RESULTS = []
for model in MODELS:
    for text, cat, pri in EVAL:
        out, secs, u = classify(model, text)
        RESULTS.append({"model": model, "text": text, "true_cat": cat, "pred_cat": out["category"], "true_pri": pri, "pred_pri": out["priority"],
                        "confidence": out["confidence"], "seconds": secs, "in": u.input_tokens, "out": u.output_tokens})
res = pd.DataFrame(RESULTS)
summary = res.assign(cat_ok=res.true_cat == res.pred_cat, pri_ok=res.true_pri == res.pred_pri).groupby("model").agg(
    category_accuracy=("cat_ok", "mean"), priority_accuracy=("pri_ok", "mean"),
    p50_latency_s=("seconds", lambda s: float(np.percentile(s, 50))), p95_latency_s=("seconds", lambda s: float(np.percentile(s, 95))),
    avg_in=("in", "mean"), avg_out=("out", "mean")).round(3)
summary

## 3. Cost per request
Paste current prices (USD per million tokens) from the official pricing page. The notebook never hard-codes prices.

In [ ]:
PRICES = {m: (None, None) for m in MODELS}     # e.g. PRICES[config.MODEL] = (input_price, output_price)
def per_request_cost(row, model):
    p_in, p_out = PRICES.get(model, (None, None))
    if p_in is None or p_out is None: return None
    return (row.avg_in * p_in + row.avg_out * p_out) / 1e6
summary["cost_per_request_usd"] = [per_request_cost(summary.loc[m], m) for m in summary.index]
summary["cost_per_1k_usd"] = summary["cost_per_request_usd"].apply(lambda c: None if c is None else round(c * 1000, 4))
summary

## 4. Decide: the cheapest model that passes

In [ ]:
passes = summary[(summary.category_accuracy >= QUALITY_BAR["category_accuracy"]) & (summary.p95_latency_s <= QUALITY_BAR["p95_latency_s"])]
if passes.empty:
    print("No model meets the bar. Improve the prompt or the eval set, or revisit the bar.")
else:
    if passes["cost_per_request_usd"].notna().all():
        choice = passes["cost_per_request_usd"].astype(float).idxmin()
    else:
        choice = next(m for m in MODELS if m in passes.index)   # MODELS is ordered cheapest tier first
    print("Choose:", choice); print("Because:", passes.loc[choice, ["category_accuracy", "p95_latency_s"]].to_dict())

## 5. Error analysis
Accuracy hides patterns. Look at what each model got wrong, especially the hard cases.

In [ ]:
wrong = res[res.true_cat != res.pred_cat][["model", "text", "true_cat", "pred_cat", "confidence"]]
wrong.sort_values(["model", "confidence"])

Low confidence on wrong answers is good news: it means confidence can drive routing (Lab 5B). High-confidence mistakes need prompt or label fixes.

## You should now have
- [ ] A written quality bar
- [ ] A comparison table with accuracy, latency percentiles, and tokens
- [ ] Cost per request from your own prices
- [ ] A decision with its reason, plus an error analysis

## Check yourself
1. Did the cheaper model pass? By how much?
2. Which hard cases did both models miss, and why?
3. How large should the eval set be before a real decision?